# 🚗 Car Evaluation System — Jupyter Notebook

> **CodTech IT Solutions — ML Internship Project**  
> **Intern:** Mayank | **ID:** CTTS148 | **Duration:** 4 Weeks  
> **Degree:** B.Tech AIML (5th Semester)

---

## Objective
Classify cars into **Unacceptable**, **Acceptable**, **Good**, or **Very Good** using the UCI Car Evaluation Dataset.  
We compare Logistic Regression, Decision Tree, Random Forest, and XGBoost.

---

## Table of Contents
1. [Import Libraries](#1)
2. [Load Dataset](#2)
3. [Data Inspection](#3)
4. [Data Cleaning](#4)
5. [Exploratory Data Analysis (EDA)](#5)
6. [Feature Encoding](#6)
7. [Train/Test Split](#7)
8. [Model Training](#8)
9. [Model Evaluation](#9)
10. [Feature Importance](#10)
11. [Model Comparison](#11)
12. [Conclusion](#12)

<a id='1'></a>
## 1. Import Libraries

In [ ]:
# Standard library imports
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path so we can import from src/
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

# Data manipulation
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
%matplotlib inline

# Machine Learning
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# XGBoost (optional)
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print('XGBoost loaded successfully!')
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed. Skipping.')

import joblib
print('All libraries imported successfully!')

<a id='2'></a>
## 2. Load Dataset

In [ ]:
# Load the UCI Car Evaluation Dataset
# Source: https://archive.ics.uci.edu/ml/datasets/Car+Evaluation

DATA_PATH = os.path.join('..', 'data', 'car.csv')

# Read the CSV file (already has a header row)
df = pd.read_csv(DATA_PATH)

print(f'Dataset Shape: {df.shape}')
print(f'\nDataset loaded from: {DATA_PATH}')
df.head()

<a id='3'></a>
## 3. Data Inspection

In [ ]:
# ── Data Types & Non-Null Counts ─────────────────────────────────────────────
print('Data Types and Non-Null Counts:')
print('=' * 45)
df.info()

In [ ]:
# ── Missing Values ────────────────────────────────────────────────────────────
missing = df.isnull().sum()
print('Missing Values Per Column:')
print(missing)
print(f'\nTotal Missing Values: {missing.sum()}')

In [ ]:
# ── Duplicate Rows ─────────────────────────────────────────────────────────────
duplicates = df.duplicated().sum()
print(f'Number of Duplicate Rows: {duplicates}')

In [ ]:
# ── Unique Values Per Feature ──────────────────────────────────────────────────
print('Unique Values Per Feature:')
print('=' * 45)
for col in df.columns:
    print(f'{col:12s}: {df[col].unique().tolist()}')

In [ ]:
# ── Target Class Distribution ──────────────────────────────────────────────────
print('Target Class Distribution:')
class_dist = df['class'].value_counts()
print(class_dist)
print(f'\nClass Proportions:')
print(df['class'].value_counts(normalize=True).round(4) * 100)

<a id='4'></a>
## 4. Data Cleaning

In [ ]:
# ── Handle Missing Values ─────────────────────────────────────────────────────
# The UCI Car Evaluation dataset has no missing values.
# We demonstrate the technique anyway.

missing_count = df.isnull().sum().sum()
print(f'Missing values found: {missing_count}')

if missing_count > 0:
    df = df.fillna(method='ffill')   # forward-fill for categorical data
    print('Applied forward-fill to handle missing values.')
else:
    print('No missing values — dataset is already clean!')

print(f'Missing values after handling: {df.isnull().sum().sum()}')

In [ ]:
# ── Remove Duplicates ─────────────────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
print(f'Rows before: {before}, Rows after: {after}')
print(f'Duplicates removed: {before - after}')

<a id='5'></a>
## 5. Exploratory Data Analysis (EDA)

In [ ]:
# ── Colour Palette ───────────────────────────────────────────────────────────
CLASS_COLORS = {'unacc': '#E63946', 'acc': '#4361EE', 'good': '#2DC653', 'vgood': '#F9844A'}
CLASS_ORDER  = ['unacc', 'acc', 'good', 'vgood']
CLASS_LABELS = {'unacc': 'Unacceptable', 'acc': 'Acceptable', 'good': 'Good', 'vgood': 'Very Good'}

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')

In [ ]:
# ── Plot 1: Class Distribution Bar Chart ─────────────────────────────────────
counts = df['class'].value_counts().reindex(CLASS_ORDER)
colors = [CLASS_COLORS[c] for c in CLASS_ORDER]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(CLASS_ORDER, counts.values, color=colors, edgecolor='white', width=0.55)

for bar, cnt in zip(bars, counts.values):
    pct = 100 * cnt / counts.sum()
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{cnt}\n({pct:.1f}%)', ha='center', fontsize=10, fontweight='bold')

ax.set_xticklabels(['Unacceptable', 'Acceptable', 'Good', 'Very Good'])
ax.set_title('Target Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../visualizations/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2: Class Distribution Pie Chart ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
labels  = [CLASS_LABELS[c] for c in CLASS_ORDER]
colors  = [CLASS_COLORS[c] for c in CLASS_ORDER]

ax.pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
       startangle=140, explode=[0.05]*4, textprops={'fontsize': 11})
ax.set_title('Class Distribution (Pie Chart)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/02_class_distribution_pie.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: Feature Value Distributions ──────────────────────────────────────
FEATURE_COLS = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    counts_feat = df[col].value_counts().sort_index()
    axes[i].bar(counts_feat.index, counts_feat.values, color='#4361EE',
                edgecolor='white', alpha=0.85)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    for j, val in enumerate(counts_feat.values):
        axes[i].text(j, val + 3, str(val), ha='center', fontsize=9)

fig.suptitle('Feature Value Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/03_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 5: Buying Price vs Class ─────────────────────────────────────────────
buying_order = ['low', 'med', 'high', 'vhigh']
ct = pd.crosstab(df['buying'], df['class'])[CLASS_ORDER].reindex(buying_order)

fig, ax = plt.subplots(figsize=(10, 6))
ct.plot(kind='bar', ax=ax, color=[CLASS_COLORS[c] for c in CLASS_ORDER],
        edgecolor='white', rot=0)
ax.set_title('Buying Price vs Car Class', fontsize=14, fontweight='bold')
ax.set_xlabel('Buying Price'); ax.set_ylabel('Count')
ax.set_xticklabels(['Low', 'Medium', 'High', 'Very High'])
ax.legend(title='Class', labels=list(CLASS_LABELS.values()))
plt.tight_layout()
plt.savefig('../visualizations/05_buying_vs_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 6: Safety Rating vs Class ───────────────────────────────────────────
safety_order = ['low', 'med', 'high']
ct_s = pd.crosstab(df['safety'], df['class'])[CLASS_ORDER].reindex(safety_order)

fig, ax = plt.subplots(figsize=(9, 6))
ct_s.plot(kind='bar', ax=ax, color=[CLASS_COLORS[c] for c in CLASS_ORDER],
          edgecolor='white', rot=0)
ax.set_title('Safety Rating vs Car Class', fontsize=14, fontweight='bold')
ax.set_xlabel('Safety Rating'); ax.set_ylabel('Count')
ax.set_xticklabels(['Low', 'Medium', 'High'])
ax.legend(title='Class', labels=list(CLASS_LABELS.values()))
plt.tight_layout()
plt.savefig('../visualizations/06_safety_vs_class.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='6'></a>
## 6. Feature Encoding

In [ ]:
# ── Define Ordinal Category Orders ────────────────────────────────────────────
# We preserve the natural ordering of each feature (low < med < high < vhigh)

FEATURE_CATEGORIES = {
    'buying':   ['low', 'med', 'high', 'vhigh'],
    'maint':    ['low', 'med', 'high', 'vhigh'],
    'doors':    ['2', '3', '4', '5more'],
    'persons':  ['2', '4', 'more'],
    'lug_boot': ['small', 'med', 'big'],
    'safety':   ['low', 'med', 'high'],
}
TARGET_ORDER = ['unacc', 'acc', 'good', 'vgood']

# Separate features and target
X = df[FEATURE_COLS].copy()
y = df['class'].copy()

# ── OrdinalEncoder for features ──────────────────────────────────────────────
categories = [FEATURE_CATEGORIES[col] for col in FEATURE_COLS]
ordinal_enc = OrdinalEncoder(categories=categories, handle_unknown='use_encoded_value', unknown_value=-1)
X_encoded = pd.DataFrame(ordinal_enc.fit_transform(X), columns=FEATURE_COLS)

print('Feature encoding complete!')
print('\nEncoded feature sample (first 5 rows):')
print(X_encoded.head())

In [ ]:
# ── LabelEncoder for target ────────────────────────────────────────────────────
label_enc = LabelEncoder()
label_enc.classes_ = np.array(TARGET_ORDER)
y_encoded = label_enc.transform(y)

print('Target class encoding:')
for i, cls in enumerate(label_enc.classes_):
    print(f'  {i} → {cls}')

print(f'\ny shape: {y_encoded.shape}')

In [ ]:
# ── Plot 4: Correlation Heatmap ──────────────────────────────────────────────
corr = X_encoded.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, linecolor='white', ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap (Encoded)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/04_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='7'></a>
## 7. Train / Test Split

In [ ]:
# Stratified 80/20 split — preserves class proportions in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f'Total samples  : {len(X_encoded)}')
print(f'Training set   : {len(X_train)} ({100*len(X_train)/len(X_encoded):.0f}%)')
print(f'Test set       : {len(X_test)} ({100*len(X_test)/len(X_encoded):.0f}%)')

<a id='8'></a>
## 8. Model Training

In [ ]:
# ── 1. Logistic Regression ────────────────────────────────────────────────────
print('Training Logistic Regression ...')
lr_model = LogisticRegression(max_iter=1000, random_state=42, multi_class='auto', solver='lbfgs')
lr_model.fit(X_train, y_train)
print('Done!')

In [ ]:
# ── 2. Decision Tree ──────────────────────────────────────────────────────────
print('Training Decision Tree ...')
dt_model = DecisionTreeClassifier(max_depth=6, min_samples_split=5, min_samples_leaf=2,
                                   criterion='gini', random_state=42)
dt_model.fit(X_train, y_train)
print('Done!')

In [ ]:
# ── 3. Random Forest ──────────────────────────────────────────────────────────
print('Training Random Forest ...')
rf_model = RandomForestClassifier(n_estimators=100, min_samples_split=5,
                                   min_samples_leaf=2, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
print('Done!')

In [ ]:
# ── 4. XGBoost (optional) ─────────────────────────────────────────────────────
xgb_model = None
if XGBOOST_AVAILABLE:
    print('Training XGBoost ...')
    xgb_model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                               eval_metric='mlogloss', random_state=42, verbosity=0)
    xgb_model.fit(X_train, y_train)
    print('Done!')
else:
    print('XGBoost not available. Skipping.')

<a id='9'></a>
## 9. Model Evaluation

In [ ]:
# ── Evaluation Helper ─────────────────────────────────────────────────────────
def evaluate(model, X_test, y_test, name, classes=None):
    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    print(f'\n── {name} ──')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1 Score  : {f1:.4f}')
    if classes:
        print(classification_report(y_test, y_pred, target_names=classes, zero_division=0))
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

class_names = list(label_enc.classes_)
results = {}

results['Logistic Regression'] = evaluate(lr_model, X_test, y_test, 'Logistic Regression', class_names)
results['Decision Tree']       = evaluate(dt_model, X_test, y_test, 'Decision Tree', class_names)
results['Random Forest']       = evaluate(rf_model, X_test, y_test, 'Random Forest', class_names)
if xgb_model:
    results['XGBoost'] = evaluate(xgb_model, X_test, y_test, 'XGBoost', class_names)

In [ ]:
# ── Confusion Matrix — Random Forest ─────────────────────────────────────────
y_pred_rf = rf_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_rf)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, linecolor='white', ax=ax)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix — Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/confusion_matrix_random_forest.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='10'></a>
## 10. Feature Importance

In [ ]:
# ── Feature Importance — Random Forest ───────────────────────────────────────
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
sorted_features = [FEATURE_COLS[i] for i in indices]
sorted_values   = importances[indices]

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.Blues_r(np.linspace(0.3, 0.85, len(sorted_features)))
ax.barh(sorted_features[::-1], sorted_values[::-1], color=colors, edgecolor='white')

for i, (feat, val) in enumerate(zip(sorted_features[::-1], sorted_values[::-1])):
    ax.text(val + 0.002, i, f'{val:.4f}', va='center', fontsize=9)

ax.set_title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Gini Importance')
plt.tight_layout()
plt.savefig('../visualizations/feature_importance_random_forest.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top Features (most important → least important):')
for f, v in zip(sorted_features, sorted_values):
    print(f'  {f:12s}: {v:.4f}')

<a id='11'></a>
## 11. Model Comparison

In [ ]:
# ── Model Comparison Bar Chart ─────────────────────────────────────────────────
metrics      = ['accuracy', 'precision', 'recall', 'f1']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
model_names  = list(results.keys())
n_models     = len(model_names)
x = np.arange(len(metrics))
bar_width = 0.18
colors    = ['#4361EE', '#3A0CA3', '#7209B7', '#F72585']

fig, ax = plt.subplots(figsize=(11, 6))
for i, (model_name, scores) in enumerate(results.items()):
    values = [scores[m] for m in metrics]
    offset = (i - n_models/2 + 0.5) * bar_width
    bars = ax.bar(x + offset, values, width=bar_width, label=model_name,
                  color=colors[i % len(colors)], edgecolor='white', alpha=0.9)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('../visualizations/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────
summary_df = pd.DataFrame(results).T.round(4)
summary_df.columns = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
print('Model Performance Summary:')
print(summary_df.to_string())

best_model_name = max(results, key=lambda k: results[k]['f1'])
print(f'\n🏆 Best Model: {best_model_name} (F1 = {results[best_model_name]["f1"]:.4f})')

In [ ]:
# ── Save Models ───────────────────────────────────────────────────────────────
os.makedirs('../models', exist_ok=True)

joblib.dump(lr_model,    '../models/logistic_regression.pkl')
joblib.dump(dt_model,    '../models/decision_tree.pkl')
joblib.dump(rf_model,    '../models/random_forest.pkl')
joblib.dump(ordinal_enc, '../models/ordinal_encoder.pkl')
joblib.dump(label_enc,   '../models/label_encoder.pkl')
if xgb_model:
    joblib.dump(xgb_model, '../models/xgboost.pkl')

print('All models saved to models/ directory!')

<a id='12'></a>
## 12. Conclusion

### Key Findings

1. **Best Model:** Random Forest achieved the highest accuracy and F1 score (~97%)
2. **Most Important Feature:** Safety rating — cars with `safety=low` are almost always Unacceptable
3. **Class Imbalance:** The dataset is skewed (~70% Unacceptable), but stratified splitting ensures fair evaluation
4. **Ordinal Encoding:** Using `OrdinalEncoder` with explicit category order significantly helps tree-based models
5. **Logistic Regression Limitation:** A linear model (~85%) cannot fully capture the non-linear decision boundaries

### Recommendations
- Use **Random Forest** for deployment (best accuracy + reliability)
- **Safety** and **persons** are the top 2 most influential features
- For real-world use, collect more `good` and `vgood` samples to address class imbalance

---

**Intern:** Mayank | **ID:** CTTS148 | **CodTech IT Solutions**